# Jetbot System Self-check

Run all cells to check Jetbot services, internet access, Robot API, Camera API, and key files.

This notebook does not move the robot by default.

In [ ]:
import glob
import json
import os
import platform
import subprocess
import sys
import time
import urllib.request

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def run_cmd(cmd, timeout=8):
    try:
        p = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
            universal_newlines=True,
        )
        return {
            "ok": p.returncode == 0,
            "returncode": p.returncode,
            "stdout": p.stdout.strip(),
            "stderr": p.stderr.strip(),
        }
    except Exception as exc:
        return {"ok": False, "returncode": None, "stdout": "", "stderr": repr(exc)}

def check_service(name):
    result = run_cmd(["systemctl", "is-active", name], timeout=5)
    return {
        "ok": result["ok"],
        "service": name,
        "state": result["stdout"] or result["stderr"],
    }

def check_internet(url="https://www.baidu.com", timeout=8):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return {"ok": True, "url": url, "status": resp.status}
    except Exception as exc:
        return {"ok": False, "url": url, "error": repr(exc)}

def find_paths(patterns, roots=("/home/jetbot",)):
    found = {}
    for pattern in patterns:
        matches = []
        for root in roots:
            matches.extend(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        found[pattern] = sorted(set(matches))[:30]
    return {"ok": any(found[p] for p in found), "found": found}

def check_robot():
    try:
        from jetbot import Robot
        robot = Robot()
        robot.stop()
        return {"ok": True, "import_ok": True, "instantiated": True, "stop_ok": True}
    except Exception as exc:
        return {"ok": False, "error": repr(exc)}

def restart_nvargus(wait_seconds=4):
    result = run_cmd(["sudo", "systemctl", "restart", "nvargus-daemon"], timeout=12)
    time.sleep(wait_seconds)
    return result

def _check_camera_once(width=224, height=224, image_path="selfcheck_frame.jpg"):
    camera = None
    try:
        from jetbot import Camera
        camera = Camera.instance(width=width, height=height)
        time.sleep(2)
        frame = camera.value
        if frame is None:
            return {"ok": False, "error": "camera.value is None"}
        saved = False
        try:
            import cv2
            saved = bool(cv2.imwrite(image_path, frame))
        except Exception as exc:
            return {"ok": False, "error": "cv2 save failed: " + repr(exc)}
        return {
            "ok": True,
            "frame_shape": list(frame.shape),
            "image_saved": saved,
            "image_path": image_path if saved else None,
            "camera_released": True,
        }
    except Exception as exc:
        return {
            "ok": False,
            "error": repr(exc),
            "hint": "Try: sudo systemctl stop jetbot_start; sudo systemctl restart nvargus-daemon; restart kernel.",
        }
    finally:
        if camera is not None:
            try:
                camera.stop()
            except Exception:
                pass

def check_camera(width=224, height=224, image_path="selfcheck_frame.jpg"):
    first = _check_camera_once(width=width, height=height, image_path=image_path)
    if first.get("ok"):
        first["attempts"] = 1
        return first
    restart = restart_nvargus(wait_seconds=5)
    second = _check_camera_once(width=width, height=height, image_path=image_path)
    second["attempts"] = 2
    second["first_error"] = first.get("error")
    second["nvargus_restart"] = restart
    if not second.get("ok"):
        second["hint"] = "Camera still failed after retry. Shutdown other camera notebooks/kernels, run sudo systemctl restart nvargus-daemon in terminal, then restart this kernel."
    return second

def write_text_report(report, path):
    lines = ["Jetbot System Self-check", "Time: " + report["time"], ""]
    lines.append("Summary: " + report["summary"]["message"])
    lines.append("")
    for name, value in report["checks"].items():
        lines.append("== {} ==".format(name))
        lines.append(json.dumps(value, ensure_ascii=False, indent=2))
        lines.append("")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

print("helpers loaded")

In [ ]:
# Optional: stop official app service before camera/robot checks.
# This is temporary. It does not disable the service permanently.
stop_service = True

if stop_service:
    print(run_cmd(["sudo", "systemctl", "stop", "jetbot_start"], timeout=10))
print("jetbot_start:", check_service("jetbot_start"))

In [ ]:
# If camera failed recently, restarting nvargus-daemon usually fixes it.
# Keep this enabled for self-check.
restart_camera_service = True

if restart_camera_service:
    print(restart_nvargus(wait_seconds=5))
print("nvargus-daemon:", check_service("nvargus-daemon"))

In [ ]:
report = {"time": now(), "checks": {}}
checks = report["checks"]

checks["python"] = {
    "ok": True,
    "version": sys.version.replace("\n", " "),
    "executable": sys.executable,
    "platform": platform.platform(),
}
checks["jetbot_start"] = check_service("jetbot_start")
checks["nvargus_daemon"] = check_service("nvargus-daemon")
checks["jetbot_jupyter"] = check_service("jetbot_jupyter")
checks["dev_video"] = {"ok": len(glob.glob("/dev/video*")) > 0, "devices": sorted(glob.glob("/dev/video*"))}
checks["internet"] = check_internet()
checks["files"] = find_paths(["robot.py", "RGB_Lib.py", "servoserial.py", "*.ipynb"])
checks["robot"] = check_robot()
checks["camera"] = check_camera(width=224, height=224, image_path="selfcheck_frame.jpg")

failed = [name for name in ["python", "robot", "camera", "internet"] if not checks.get(name, {}).get("ok")]
report["summary"] = {
    "ok": not failed,
    "message": "Core checks passed: Python, Robot, Camera, Internet." if not failed else "Core checks failed: " + ", ".join(failed),
}

with open("jetbot_selfcheck_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
write_text_report(report, "jetbot_selfcheck_report.txt")

print(report["summary"]["message"])
print("JSON report: jetbot_selfcheck_report.json")
print("Text report: jetbot_selfcheck_report.txt")
if checks["camera"].get("image_saved"):
    print("Camera frame:", checks["camera"].get("image_path"))

report

## Optional motion test

Run the next cell only when the tracks are lifted or the area is clear.

In [ ]:
# OPTIONAL MOTION TEST. Uncomment and run only when safe.
# import time
# from jetbot import Robot
# robot = Robot()
# robot.forward(0.35)
# time.sleep(0.35)
# robot.stop()
# print("motion test done")